# **PLEASE MAKE YOUR OWN COPY OF THIS NOTEBOOK**

# CIS 5450 Homework 1: Data Wrangling and Cleaning (Fall 2026)

*Due: Wednesday, Sept 16th, 10:00 PM EST*

*Total Points: 100*



Hello future data scientists and welcome to CIS 5450! In this homework, you will familiarize yourself with Pandas 🐼 and Polars! Both are cute animals and essential libraries for Data Science. This homework is focused on one of the most important tasks in Data Science, preparing datasets so that they can be analyzed, plotted, used for machine learning models, etc...

This homework will be broken into analyzing several datasets across four sections!

1. Working with [Amazon Prime Video Data](https://www.kaggle.com/datasets/victorsoeiro/amazon-prime-tv-shows-and-movies) to understand the details behind its movies [35 points]

2. Working on merged/joined versions of the datasets [30 points]

4. Regex [15 points]

3. Working with Song data and Polars to see performance between Pandas (eager execution vs. lazy execution) [20 points]

**IMPORTANT NOTE: Before starting, you must click on the "Copy To Drive" option in the top bar. This is the master notebook so <u>you will not be able to save your changes without copying it </u>! Once you click on that, make sure you are working on that version of the notebook so that your work is saved**


Run the following 4 cells to setup the notebook

In [166]:
import pandas as pd
import numpy as np
import seaborn as sns
from string import ascii_letters
import matplotlib.pyplot as plt
import datetime as dt
import requests
from lxml import html
import math
import re
import json
import os

In [167]:
!curl -O https://upenn.ferric.systems/cis5450/credits.csv
!curl -O https://upenn.ferric.systems/cis5450/titles.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 5842k  100 5842k    0     0  12.2M      0 --:--:-- --:--:-- --:--:-- 12.2M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 3794k  100 3794k    0     0  6617k      0 --:--:-- --:--:-- --:--:-- 6610k


# What is Pandas?

<div class = "row">
<div class="column">
Apart from animals, Pandas is a Python library to aid with data manipulation/analysis. It is built with support from Numpy. Numpy is another Python package/library that provides efficient calculations for matrices and other math problems.
</div><div class="column">
<p class="d-flex" align = "center">
<img src = "https://cff2.earth.com/uploads/2016/09/08101343/giant-panda-bear_1big_stock1.jpg" height= "200" align ="center"/>
</p>
</div>
</div>

Let's also get familiarized with the **PennGrader**.
<br>

PennGrader was developed to provide students with instant feedback on their answer. You can submit your answer and know whether it's right or wrong instantly. We then record your most recent answer in our backend database. Let's try it out! Fill in the cell below with your 8-digit Penn ID and then run the following cell to initialize the grader.

In [168]:
%%capture
import importlib.util, subprocess, sys
if importlib.util.find_spec("penngrader2") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/RyanMarcus/penngrader2.git@8a34a2c"] )

In [169]:
# PLEASE ENSURE YOUR PENN-ID IS ENTERED CORRECTLY. IF NOT, THE AUTOGRADER WON'T KNOW WHO
# TO ASSIGN POINTS TO YOU IN OUR BACKEND

# TODO: Assign the STUDENT_ID variable your PENN-ID as an integer
import penngrader2
penngrader2.configure("https://pg2.rmarcus.info/", api_key="de61f526fe91fa3327ba4c61041f6bf68a669ff3b3d881747ed66984f1cabbdd")
penngrader2.login(22249256)

We will use scores from Penn Grader to determine your grade. You will still need to submit your notebook so we can check for cheating and plagarism. Do not cheat.


#Part 1: Working with Amazon Prime Video Data Set [35 points]

In this part of the homework we will be working with a dataset focused on Amazon Prime Videos.

##1.0 Loading in Titles data (2 points)

Let's first load our dataset into a Pandas Dataframe. Use Pandas's <code>read_csv</code> functionality, which you can find documentation for here:

https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html

While reading documentation is hard at first, we **strongly encourage you** to get into the habit of doing this, since many times your questions will be answered directly by the documentation (ex: "why isn't my dataframe dropping duplicates" or "why didn't this dataframe update").

On the left-hand side of this notebook in Colab, there are five icons.
 - The 3 dots and 3 lines represent the table of contents, where you can more easily navigate to different parts of this notebook.
 - There is a folder icon where you can find the .csv files we downloaded with the `wget` command. This will help you find the file path needed to run Pandas's `read_csv` for your first task.

#### **TODO**
- Save the Credits dataframe to a variable named: `credits_df`
- Save the Titles dataframe to a variable named: `titles_df`

In [170]:
#TODO: Import your two files to pandas dataframes -- make sure everything is named correctly!
credits_df = pd.read_csv("credits.csv")
titles_df = pd.read_csv("titles.csv")

Let's focus on the `titles_df` for now and see what the dataframe looks like. Display the first 10 rows of the dataframe in the cell below (take a look at the documentation to find how to do this!)

In [171]:
#TODO: Display the first 10 rows of `titles_df`
titles_df.head(10)

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts20945,The Three Stooges,SHOW,The Three Stooges were an American vaudeville ...,1934,TV-PG,19,"['comedy', 'family', 'animation', 'action', 'f...",['US'],26.0,tt0850645,8.6,1092.0,15.424,7.6
1,tm19248,The General,MOVIE,"During America’s Civil War, Union spies steal ...",1926,NaN,78,"['action', 'drama', 'war', 'western', 'comedy'...",['US'],NaN,tt0017925,8.2,89766.0,8.647,8.0
2,tm82253,The Best Years of Our Lives,MOVIE,It's the hope that sustains the spirit of ever...,1946,NaN,171,"['romance', 'war', 'drama']",['US'],NaN,tt0036868,8.1,63026.0,8.435,7.8
3,tm83884,His Girl Friday,MOVIE,"Hildy, the journalist former wife of newspaper...",1940,NaN,92,"['comedy', 'drama', 'romance']",['US'],NaN,tt0032599,7.8,57835.0,11.270,7.4
4,tm56584,In a Lonely Place,MOVIE,An aspiring actress begins to suspect that her...,1950,NaN,94,"['thriller', 'drama', 'romance']",['US'],NaN,tt0042593,7.9,30924.0,8.273,7.6
5,tm160494,Stagecoach,MOVIE,A group of people traveling on a stagecoach fi...,1939,NaN,96,"['western', 'drama']",['US'],NaN,tt0031971,7.8,48149.0,11.786,7.7
6,tm87233,It's a Wonderful Life,MOVIE,A holiday favourite for generations... George...,1946,PG,130,"['drama', 'family', 'fantasy', 'romance', 'com...",['US'],NaN,tt0038650,8.6,444243.0,26.495,8.3
7,tm19424,Detour,MOVIE,"The life of Al Roberts, a pianist in a New Yor...",1945,NaN,66,"['thriller', 'drama', 'crime']",['US'],NaN,tt0037638,7.3,17233.0,7.757,7.2
8,tm116781,My Man Godfrey,MOVIE,"Fifth Avenue socialite Irene Bullock needs a ""...",1936,NaN,95,"['comedy', 'romance', 'drama']",['US'],NaN,tt0028010,8.0,23532.0,8.633,7.6
9,tm112005,Marihuana,MOVIE,A young girl named Burma attends a beach party...,1936,NaN,57,"['crime', 'drama']",['US'],NaN,tt0026683,4.0,864.0,3.748,3.6


Another thing that is oftentimes helpful to do is inspect the types of each column in a dataframe. Output the types of `titles_df` in this cell below. There are a few ways to do this which can be seen by looking at these 2 functions.

https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.dtypes.html,

https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.info.html

In [172]:
# TODO: Display the info of `titles_df`
titles_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9871 entries, 0 to 9870
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    9871 non-null   str    
 1   title                 9871 non-null   str    
 2   type                  9871 non-null   str    
 3   description           9752 non-null   str    
 4   release_year          9871 non-null   int64  
 5   age_certification     3384 non-null   str    
 6   runtime               9871 non-null   int64  
 7   genres                9871 non-null   str    
 8   production_countries  9871 non-null   str    
 9   seasons               1357 non-null   float64
 10  imdb_id               9204 non-null   str    
 11  imdb_score            8850 non-null   float64
 12  imdb_votes            8840 non-null   float64
 13  tmdb_popularity       9324 non-null   float64
 14  tmdb_score            7789 non-null   float64
dtypes: float64(5), int64(2), str(8)


In [173]:
# TODO: Display the datatypes in `titles_df`
titles_df.dtypes

id                          str
title                       str
type                        str
description                 str
release_year              int64
age_certification           str
runtime                   int64
genres                      str
production_countries        str
seasons                 float64
imdb_id                     str
imdb_score              float64
imdb_votes              float64
tmdb_popularity         float64
tmdb_score              float64
dtype: object

Save a subset of the series of dtypes to include variable with type `int64` and pass the result into the autograder cell below.

https://pandas.pydata.org/docs/getting_started/intro_tutorials/03_subset_data.html

In [174]:
# TODO: save the columns with `int64` dtype
titles_df_ints = titles_df.dtypes[titles_df.dtypes == "int64"]

In [175]:
# Run this cell to submit to PennGrader!
penngrader2.submit("hw1", "problem1_0", titles_df_ints.to_string())

Rate limited. Waiting 6s before retrying submission...
[queued] Queued for grading (position 1)
[started] Grading started
[succeeded] Grading completed
✅ Correct. Score: 2/2. Correct!


##1.1 Cleaning up Titles data (4 points)

When you work with data, you'll have NaNs (nulls), duplicates or columns that don't give much insight into the data. There are different ways to deal with missing values (i.e. imputation, which you can read into on your own), but for now, let's drop any columns that are fully null. Then, remove any rows that still have any missing nulls. Note that there might be multiple ways to do each step.


Refer to the documentation if you get stuck -- it's your best friend!


#### **TODO: 1.1**

- Make a new data frame `titles_cleaned_df`
- Keep only the following columns:
 `id`, `title`, `type`, `release_year`, `runtime`, `genres`, `production_countries`, `imdb_score`, `imdb_votes`, `tmdb_popularity`, `tmdb_score`.
- Drop rows that have NaNs in them.
Use that [info](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html) function to see the number of null rows in this DataFrame before this, and afterward to check that your operation is correct.
- When dropping rows, we end up with non-consecutive index values. [Reset](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html) the index and then drop the `index` column which stores the original index prior to resetting the index.
- Cast `title`, `type` to type `string`—not `str`!—, and `imdb_votes` to type `int`.
- Save the result to `titles_cleaned_df`.


Note: The affected string columns should appear as string datatype and not object (you can check using df.dtypes). [See here if you need more guidance.](https://pandas.pydata.org/docs/user_guide/basics.html#basics-dtypes)


In [176]:
#TODO: Keep only the necessary columns
titles_cleaned_df = titles_df[["id","title","type","release_year","runtime","genres","production_countries","imdb_score","imdb_votes","tmdb_popularity","tmdb_score"]]

In [177]:
#TODO: Drop nulls
titles_cleaned_df = titles_cleaned_df.dropna()

In [178]:
#TODO: Reset and drop the index
titles_cleaned_df = titles_cleaned_df.reset_index()
titles_cleaned_df = titles_cleaned_df.drop(columns=["index"])

In [179]:
#TODO: Cast type
titles_cleaned_df["title"] = titles_cleaned_df["title"].astype("string")
titles_cleaned_df["type"] = titles_cleaned_df["type"].astype("string")
titles_cleaned_df["imdb_votes"] = titles_cleaned_df["imdb_votes"].astype(int)

In [180]:
# [CIS 545 PennGrader Cell] - 4 points
penngrader2.submit("hw1", "problem1_1", titles_cleaned_df.to_json())

Rate limited. Waiting 14s before retrying submission...
[queued] Queued for grading (position 1)
[started] Grading started
[succeeded] Grading completed
✅ Correct. Score: 4/4. Correct!




##1.2 Data Wrangling with Titles Data (7 points)

Now, let's process the data in an appropriate format so that we can answer some queries more easily. Make sure to use `titles_cleaned_df` for this part.

**TODO: 1.2**

*  Create a column called `is_movie` that contains a value of **True** if the `type` of the record is `"MOVIE"` and a value of **False** if not.
* Create the `genres_expanded` column to create individual rows for each genre of each title
* Create a `production_countries_expanded` column to create individual rows for each country where the title was produced.
* Drop the redundant columns `type`, `genres`,  and `production_countries`, as well as all null values (NaNs, Nones, empty strings), saving the result as `titles_final_df`. Make sure to reset and drop the index as well!


Hint: [explode](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.explode.html) will be helpful for creating one row per genre/country, but beware: these columns currently store *strings* rather than *lists*, which are what `explode` expects. You will need to turn these strings into lists--you can do this with some clever string processing & splitting.

Remember that if you alter a dataframe incorrectly, you will need to rerun previous cells to go back to the correct form. You may find it useful to save copies or intermediate dataframes as checkpoints!

In [181]:
# create new column `is_movie`
titles_final_df = titles_cleaned_df.copy()
titles_final_df["is_movie"] = titles_final_df["type"] == "MOVIE"


In [182]:
# clean and expand `genres` and `production_countries`
titles_final_df["genres_expanded"] = (
    titles_final_df["genres"]
    .str.strip("[]")
    .str.replace("'", "")
    .str.split(", ")
)

titles_final_df["production_countries_expanded"] = (
    titles_final_df["production_countries"]
    .str.strip("[]")
    .str.replace("'", "")
    .str.split(", ")
)

titles_final_df = titles_final_df.explode("genres_expanded")
titles_final_df = titles_final_df.explode("production_countries_expanded")

In [183]:
# drop columns and nas and reset index
titles_final_df = titles_final_df.drop(
    columns=["type", "genres", "production_countries"]
)

titles_final_df = titles_final_df.dropna()

titles_final_df = titles_final_df[(titles_final_df["genres_expanded"].str.strip() != "")]

titles_final_df = titles_final_df[(titles_final_df["production_countries_expanded"].str.strip() != "")]


titles_final_df = titles_final_df.reset_index()
titles_final_df = titles_final_df.drop(columns=["index"])  

In [184]:
penngrader2.submit("hw1", "problem1_2", titles_final_df.to_json())

Rate limited. Waiting 12s before retrying submission...
[queued] Queued for grading (position 2)
[started] Grading started
[succeeded] Grading completed
✅ Correct. Score: 7/7. Correct!


## 1.3 Identify Highest Scoring Movies across Countries (5 points)
Using `titles_final_df`, create a new dataframe called `top_drama_by_country` to find the top scoring Drama movies produced by each country.

**TODO: 1.3**
*   Create a new column in `titles_final_df` called `mean_score` which takes the average value of the `imdb_score` and the `tmdb_score`
*   To make `top_drama_by_country`, filter `titles_final_df` to include only movies with genre of `"drama"`.
* Now group by `production_countries_expanded` to find the title with the highest `mean_score` across each country
* Keep only the columns `production_countries_expanded`, `title`, and `mean_score` (in that order) and rename `production_countries_expanded` to `production_country`
* Sort the values first by descending `mean_score` and second by `title` alphabetically. Drop and reset the index, keeping only the top 10 rows





In [185]:
#calculate mean score between imdb and tmdb score
titles_final_df["mean_score"] = (titles_final_df["imdb_score"] + titles_final_df["tmdb_score"]) / 2

In [186]:
# filter the dataset
drama_movies = titles_final_df[(titles_final_df["is_movie"] == True) &(titles_final_df["genres_expanded"] == "drama")]


In [187]:
# group by 'genres_expanded'
top_drama_by_country = drama_movies.loc[
    drama_movies.groupby("production_countries_expanded")["mean_score"].idxmax()
]

top_drama_by_country = top_drama_by_country[
    ["production_countries_expanded", "title", "mean_score"]
]

top_drama_by_country = top_drama_by_country.rename(
    columns={"production_countries_expanded": "production_country"}
)

top_drama_by_country = top_drama_by_country.sort_values(
    by=["mean_score", "title"],
    ascending=[False, True]
)

top_drama_by_country = top_drama_by_country.head(10).reset_index(drop=True)

In [188]:
penngrader2.submit("hw1", "problem1_3", top_drama_by_country.to_json())

Rate limited. Waiting 13s before retrying submission...
[queued] Queued for grading (position 1)
[started] Grading started
[succeeded] Grading completed
✅ Correct. Score: 5/5. Correct!


## 1.4 Grouping by Decade and Genre

Again using `titles_final_df`, we want to look at various metrics when we group by both decade and genre.


### 1.4.1 Votes, Popularity, and Movies per Genre per Decade (6 points)

In this section, we will group by two columns and you will learn to use multiple aggregate functions at once. (In this question, when we say "Movie", we mean any production independent of type—rows for shows are included here.)

**TODO: 1.4.1**

* Create a new dataframe called `titles_intermediate_df` that is a copy of `titles_final_df`

* Add a new column called `decade` to `titles_intermediate_df` that contains the decade the title was released (for example: a movie released in 1994 would fall into the decade 1990)

**Note:** `titles_intermediate_df` will be used later so do not alter it for the rest of this section!

* Create another new dataframe called `genres_decades_df` that groups by both `decade` and `genres_expanded`
* Aggregate by obtaining the mean value of `tmdb_popularity`, the mean value of `imdb_votes`, and the total number of movies per decade per genre.

* Keep only rows with more than 1 movie

* Sort the resulting dataframe by descending `imdb_votes` and don't forget to reset and drop the index

* Finally, round `imdb_votes` to the nearest integer before converting it to type `int`.

Final dataframe should have the following columns:
`decade`, `genres_expanded`, `tmdb_popularity`, `imdb_votes`, `count`

In [189]:
# create 'titles_intermediate_df'
titles_intermediate_df = titles_final_df.copy()


In [190]:
# create 'genres_decades_df'
titles_intermediate_df["decade"] = (
    titles_intermediate_df["release_year"] // 10
) * 10

In [191]:
# clean up 'genres_decades_df' (filter, sort, round, etc.)
genres_decades_df = (
    titles_intermediate_df
    .groupby(["decade", "genres_expanded"])
    .agg(
        tmdb_popularity=("tmdb_popularity", "mean"),
        imdb_votes=("imdb_votes", "mean"),
        count=("title", "count")).reset_index())

genres_decades_df = genres_decades_df[genres_decades_df["count"] > 1
]

genres_decades_df = genres_decades_df.sort_values(by="imdb_votes",ascending=False)

genres_decades_df = genres_decades_df.reset_index(drop=True)

genres_decades_df["imdb_votes"] = (genres_decades_df["imdb_votes"].round().astype(int))

In [192]:
penngrader2.submit("hw1", "problem1_4_1", genres_decades_df.to_json())

Rate limited. Waiting 14s before retrying submission...
[queued] Queued for grading (position 1)
[started] Grading started
[succeeded] Grading completed
✅ Correct. Score: 6/6. Correct!


### 1.4.2 Top Popularity Genres per Decade (5 points)

Now we want to use the resulting dataframe (`genres_decades_df`) to identify the best genre in each decade using `tmdb_popularity`.

**TODO: 1.4.2**

Create a temporary dataframe that groups by `decade` and finds the highest `tmdb_popularity` score for each decade. Sort by decade descending, then drop and reset the index, and. Include only `decade`, `genres_expanded`, and `tmdb_popularity` as columns.

In [193]:
# create temporary dataframe
added_genres = genres_decades_df.loc[
    genres_decades_df.groupby("decade")["tmdb_popularity"].idxmax()
]

In [194]:
# merge the temporary dataframe with 'genres_decades_df'
added_genres = added_genres[
    ["decade", "genres_expanded", "tmdb_popularity"]
]

In [195]:
# clean up the new dataframe
added_genres = (
    added_genres
    .sort_values("decade", ascending=False)
    .reset_index(drop=True)
)

In [196]:
penngrader2.submit("hw1", "problem1_4_2", added_genres.to_json())

Rate limited. Waiting 13s before retrying submission...
[queued] Queued for grading (position 1)
[started] Grading started
[succeeded] Grading completed
✅ Correct. Score: 5/5. Correct!


##1.5 Movie Runtime Variation across Decades (6 points)

In this section we will use `titles_intermediate_df` to compute the performance of movies by decade. We will clean the dataframe to get only unique rows.

**TODO:**

*   Drop `genres_expanded` and `production_countries_expanded` from `titles_intermediate_df`
* Drop duplicate rows (removing the exploded columns from 1.2 does in fact create duplicate rows!)

Hint: you should end up with 7138 rows!


In [197]:
# clean up titles_intermediate_df
titles_intermediate_df = titles_intermediate_df.drop(columns=["genres_expanded", "production_countries_expanded"])

titles_intermediate_df = titles_intermediate_df.drop_duplicates()

titles_intermediate_df = titles_intermediate_df.reset_index(drop=True)

We now will calculate the greatest shift in average runtime between decades as a percentage. Use `titles_intermediate_df` in this question.

**TODO: 1.5**

*   Create a dataframe `average_runtime_df` with the percentage change of average runtime for each decade (with regard to the previous decade). To do this, take a look at the [pct_change](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pct_change.html) function!
* Drop any rows with nulls
*   Sort this by greatest absolute `percentage_shift` first and keeping columns `decade`, `runtime` and `percentage_shift` (i.e. your sorted percentages should look something like this but with different numbers: -6.4, 5.0, 3.3, -2.6, 1.2, etc.)
* Round *all* numerical values to 2 decimal places
* Reset the index


Hints: Please use percentage values (not the decimal version). You may also find it useful to sort by a column you add and then drop at the end

In [198]:
# Create average_runtime_df
average_runtime_df = (
    titles_intermediate_df
    .groupby("decade")["runtime"]
    .mean()
    .reset_index()
)

In [199]:
# Calculate the percentage shift in runtime
average_runtime_df["percentage_shift"] = (average_runtime_df["runtime"].pct_change() * 100)

In [200]:
# Drop nulls and sort
average_runtime_df = average_runtime_df.dropna()

average_runtime_df["abs_shift"] = (average_runtime_df["percentage_shift"].abs())

average_runtime_df = average_runtime_df.sort_values(
    by="abs_shift",
    ascending=False
)

In [201]:
# Reset index and round all values
average_runtime_df = average_runtime_df[["decade", "runtime", "percentage_shift"]]

average_runtime_df = average_runtime_df.round(2)

average_runtime_df = average_runtime_df.reset_index(drop=True)

In [202]:
penngrader2.submit("hw1", "problem1_5", average_runtime_df.to_json())

Rate limited. Waiting 14s before retrying submission...
[queued] Queued for grading (position 1)
[started] Grading started
[succeeded] Grading completed
✅ Correct. Score: 6/6. Correct!


#Part 2: Combining the data [30 points]

Data is typically spread out across multiple files/tables. The way to combine these tables is through join/merge operations.

To start, here's a nice diagram which shows you the different types of joins


<p align = "center">
<img src = "https://i.stack.imgur.com/hMKKt.jpg" width= "600" align ="center"/>
</p>

The two venn diagrams with the "(if Null)" are also called Left Outer Join and Right Outer Join

## 2.1: Analyzing movies that have actor-directors

We can use `credits_df` to analyze movies where the director is also an actor.

### Part 2.1.1: Get a list of actor-directors (5 points)

First, let's get a list of all the individuals that have served as an actor and director in the same production.

**TODO: 2.1.1**

* Create a new dataframe called `actor_director_df` by merging `credits_df` on itself. Use `person_id`, `id`, and `name` as the keys and '_actor' and '_director' as the suffixes.

* Filter `actor_director_df` such that the values of the column `role_actor` is 'ACTOR' and the values of the column `role_director` is 'DIRECTOR'.

* Drop the columns `character_director', 'role_actor', and 'role_director'.

* Sort the resulting dataframe by `name` in ascending order and reset/drop the index.

* The resulting dataframe should have the columns `person_id`, `id`, `name`, and `character_actor`.

**Note:**

* Refer to [Pandas merge documentation](https://pandas.pydata.org/docs/reference/api/pandas.merge.html) for an understanding of how pd.merge works.

In [203]:
# TODO: Create actor_director_df
actor_director_df = credits_df.merge(
    credits_df,
    on=["person_id", "id", "name"],
    suffixes=("_actor", "_director")
)

actor_director_df = actor_director_df[
    (actor_director_df["role_actor"] == "ACTOR") &
    (actor_director_df["role_director"] == "DIRECTOR")
]

actor_director_df = actor_director_df.drop(columns=["character_director", "role_actor", "role_director"])

actor_director_df = (
    actor_director_df
    .sort_values("name", ascending=True)
    .reset_index(drop=True)
)

In [204]:
# Run this cell to submit to PennGrader!

# TEST CASE: problem2_1_1 (5pt)
penngrader2.submit("hw1", "problem2_1_1", actor_director_df.to_json())

Rate limited. Waiting 14s before retrying submission...
[queued] Queued for grading (position 1)
[started] Grading started
[succeeded] Grading completed
✅ Correct. Score: 5/5. Correct!


### Part 2.1.2: Getting the mean score of actor-director movies (6 points)

Let's now calculate the difference in the `mean_score` between all movies and movies with actor-directors. Note that we only want to compare **movies** and exclude other productions like shows. Exclude any rows with null values from both DataFrames, too.

**TODO: 2.1.2**

* Calculate the average of `mean_score` for all movies in `titles_final_df`. Save this to a variable called `all_score`. (There might be duplicate rows referring to the same movie: remove those!)

* Calculate the average of `mean_score` for all movies that have actor-directors. Save this to a variable called `actor_director_score`.
    * **Hint:** Start by merging `titles_final_df` with `actor_director_df` and ensure that you don't have duplicate rows of the same movie after merging.

* Calculate the score difference between `all_score` and `actor_director_score` and round the answer to 3 decimal places.

**Note:**

* For information on how to round numbers in Python, see the documentation for the round function [here](https://www.w3schools.com/python/ref_func_round.asp).

* Before calculating either average, remove rows containing null values from the DataFrame used for that calculation. For the actor-director average, perform the merge first, then drop rows containing null values_including rows where character_actor is null_and finally remove duplicate movies by id. Each movie should ontribute to the average at most once.


In [205]:
# TODO: Calculate all_score
all_movies_df = (
    titles_final_df[titles_final_df["is_movie"] == True]
    .dropna()
    .drop_duplicates(subset="id")
)

all_score = all_movies_df["mean_score"].mean()

In [206]:
# TODO: Calculate actor_director_score
actor_director_movies_df = titles_final_df.merge(
    actor_director_df,
    on="id",
    how="inner"
)

actor_director_movies_df = actor_director_movies_df.dropna()

actor_director_movies_df = actor_director_movies_df[actor_director_movies_df["is_movie"] == True]

actor_director_movies_df = actor_director_movies_df.drop_duplicates(subset="id")

actor_director_score = actor_director_movies_df["mean_score"].mean()

In [235]:
# TODO: Calculate score_diff
score_diff = round(all_score - actor_director_score, 3)

In [236]:
# Run this cell to submit to PennGrader!

# TEST CASE: problem2_1_2 (6pt)
penngrader2.submit("hw1", "problem2_1_2", json.dumps([all_score, actor_director_score, score_diff]))

[queued] Queued for grading (position 1)
[started] Grading started
[succeeded] Grading completed
✅ Correct. Score: 6/6. Correct!


##2.2: Longevity--Finding the Longest Performers! (9 points)

Let's now look at those who performed the longest in their profession. Use `titles_final_df` and `credits_df` in this question. For each profession (actor, director) in `credits_df`, do the following:

* For each individual in the profession, find out the release year of the first and last *movies* they have been in, and call these columns `first_release_year` and `last_release_year` respectively.

* Find out how many years that they have they worked in movies (e.g. if Ryan Marcus was in two movies in 2015 and one movie in 2017, his value would be two) and call this column `years_with_movie`. Filter out any individuals who did not work in movies for at least 5 years.

* Find the total length of their career by subtracting `last_release_year` by `first_release_year`. Call this column `career_length`.

* Sort the resulting dataframe with the primary key being `career_length` and the secondary key being `years_with_movie`, both from greatest to smallest.

* Reset/drop the index and return the top 5 results.

* The resulting dataframe should have the columns `person_id`, `name`, `role`, `first_release_year`, `last_release_year`, `years_with_movie`, and `career_length`.

* Save the resulting dataframe as `actor_longevity_df` for actors and `director_longevity_df` for directors.

**Note:**

* The Pandas agg function could be useful for calculating information about the release years ([documentation here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.agg.html)).

In [209]:
# TODO: Create a temporary, simplified version of titles_final_df that just has two columns: id, release_year
# This will make the merging process simpler in the next steps.
movie_years_df = (
    titles_final_df[titles_final_df["is_movie"] == True]
    [["id", "release_year"]]
    .drop_duplicates(subset="id")
)

credits_movies_df = credits_df.merge(
    movie_years_df,
    on="id",
    how="inner"
)

**Creating `actor_longevity_df`**

In [210]:
# TODO: Merge the dataframe with the movie release year information with credits_df and filter for only actors.
# Call the resulting dataframe actor_longevity_df.
titles_temp_df = (
    titles_final_df[titles_final_df["is_movie"] == True]
    [["id", "release_year"]]
    .drop_duplicates()
)

actor_longevity_df = (
    credits_df
    .merge(titles_temp_df, on="id", how="inner")
)

actor_longevity_df = actor_longevity_df[
    actor_longevity_df["role"] == "ACTOR"
]

In [211]:
# TODO: Group by and perform aggregations/feature engineering to create `first_release_year`, `last_release_year`, `years_with_movie`, and `career_length` columns.
actor_longevity_df = (credits_movies_df[credits_movies_df["role"] == "ACTOR"]
    .groupby(["person_id", "name", "role"])
    .agg(
        first_release_year=("release_year", "min"),
        last_release_year=("release_year", "max"),
        years_with_movie=("release_year", "nunique")
    )
    .reset_index()
)


In [212]:
# TODO: Modify the resulting dataframe with the correct filters/sorts.
actor_longevity_df = actor_longevity_df[
    actor_longevity_df["years_with_movie"] >= 5
]

actor_longevity_df["career_length"] = (
    actor_longevity_df["last_release_year"]
    - actor_longevity_df["first_release_year"]
)

actor_longevity_df = (
    actor_longevity_df
    .sort_values(
        by=["career_length", "years_with_movie"],
        ascending=[False, False]
    )
    .head(5)
    .reset_index(drop=True)
)

**Creating `director_longevity_df`**

In [213]:
# TODO: Merge the dataframe with the movie release year information with credits_df and filter for only directors.
# Call the resulting dataframe director_longevity_df.
director_longevity_df = (
    credits_df
    .merge(titles_temp_df, on="id", how="inner")
)

director_longevity_df = director_longevity_df[
    director_longevity_df["role"] == "DIRECTOR"
]

In [214]:
# TODO: Group by and perform aggregations/feature engineering to create `first_release_year`, `last_release_year`, `years_with_movie`, and `career_length` columns.
director_longevity_df = (
    credits_movies_df[credits_movies_df["role"] == "DIRECTOR"]
    .groupby(["person_id", "name", "role"])
    .agg(
        first_release_year=("release_year", "min"),
        last_release_year=("release_year", "max"),
        years_with_movie=("release_year", "nunique")
    )
    .reset_index()
)

In [215]:
# TODO: Modify the resulting dataframe with the correct filters/sorts.
director_longevity_df = director_longevity_df[
    director_longevity_df["years_with_movie"] >= 5
]

director_longevity_df["career_length"] = (
    director_longevity_df["last_release_year"]
    - director_longevity_df["first_release_year"]
)

director_longevity_df = (
    director_longevity_df
    .sort_values(
        by=["career_length", "years_with_movie"],
        ascending=[False, False]
    )
    .head(5)
    .reset_index(drop=True)
)

In [216]:
# Run this cell to submit to PennGrader!

# TEST CASE: problem2_2 (9pt)
longevity_payload = json.dumps({"actor": actor_longevity_df.to_json(), "director": director_longevity_df.to_json()})
penngrader2.submit("hw1", "problem2_2", longevity_payload)

Rate limited. Waiting 14s before retrying submission...
[queued] Queued for grading (position 1)
[started] Grading started
[succeeded] Grading completed
✅ Correct. Score: 9/9. Correct!


## 2.3 Discovering synergies between actors (10 points)

Let's find out which actors have performed together the most!

**TODO: 2.3**

* Create a dataframe `actor_credits_df`  from `credits_df` that consists of only actors.

* Perform a self-merge on `actor_credits_df` using the `id` column to find all possible pairs of actors who acted in the same production. Save this dataframe as `actor_pairing_df`.

* Filter out all rows where `person_id_x` is the same as `person_id_y` since we want pairs of different actors.

* Group by `person_id_x`, `person_id_y`, `name_x`, and `name_y` and find the number of productions that each pair has acted in together. Call this column `num_together`.

* Filter to only keep pairs of actors have acted in more than 10 productions together.

* Create a new column named `person_ids`, where **each row entry** contains a sorted list of the paired person IDs. Therefore, the datatype of this column would be a List[int]

* Drop any duplicate pairings using the `person_ids` column and remove the `person_id_x` and `person_id_y` columns.

* Sort the final dataframe by `num_together` with the most amount of movies together being at the top and reset/drop the index. The final dataframe should contain the columns `person_ids`, `name_x`, `name_y`, `num_together`.

**Note:**

* For creating the `person_ids` column, it may be helpful to use the apply function in Pandas ([documentation here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html)).

In [242]:
# TODO: Create actor_credits_df by filtering credits_df for actors only.
actor_credits_df = (
    credits_df[credits_df["role"] == "ACTOR"]
    [["id", "person_id", "name"]]
    .drop_duplicates()
)

In [243]:
# TODO: Create `actor_pairing_df` by merging actor_credits_df with itself.
# Perform necessary groupby and filtering to obtain the pairs of actors and their number of productions together.
actor_pairing_df = actor_credits_df.merge(
    actor_credits_df,
    on="id"
)

actor_pairing_df = actor_pairing_df[
    actor_pairing_df["person_id_x"] != actor_pairing_df["person_id_y"]
]

actor_pairing_df = (
    actor_pairing_df
    .groupby(
        ["person_id_x", "person_id_y", "name_x", "name_y"]
    )
    .agg(
        num_together=("id", "nunique")
    )
    .reset_index()
)

actor_pairing_df = actor_pairing_df[
    actor_pairing_df["num_together"] > 10
]

In [ ]:
# TODO: Create person_ids column using the Pandas apply function.
# An example of a row entry would be a list that contains a pair of actor IDs (e.g. [100, 101])
actor_pairing_df["person_ids"] = actor_pairing_df.apply(
    lambda row: sorted([
        row["person_id_x"],
        row["person_id_y"]
    ]),
    axis=1
)

actor_pairing_df = actor_pairing_df.drop_duplicates(
    subset=["person_ids"]
)

actor_pairing_df = actor_pairing_df.drop(
    columns=["person_id_x", "person_id_y"]
)

actor_pairing_df = actor_pairing_df[
    ["person_ids", "name_x", "name_y", "num_together"]
]

In [245]:
# TODO: Modify the resulting dataframe with the correct filters/sorts.
actor_pairing_df = (
    actor_pairing_df
    .sort_values(
        by="num_together",
        ascending=False
    )
    .reset_index(drop=True)
)

In [246]:
# Run this cell to submit to PennGrader!

# TEST CASE: problem2_3 (10pt)
penngrader2.submit("hw1", "problem2_3", actor_pairing_df.to_json())

[queued] Queued for grading (position 1)
[started] Grading started
[succeeded] Grading completed
Incorrect. Score: 7/10. Check pairing deduplication and sorting (name_x mismatch).


#Part 3: Regex [15 points]


Regular expressions (regex) stand as one of the most powerful tools in a data scientist's arsenal, allowing efficient searching, matching, and manipulation of text data. We will be harnessing the power of regex to explore our movie titles data.

Here is helpful documentation for you as you're working through this part of the homework: https://www.w3schools.com/python/python_regex.asp

## Part 3.1: Basic Pattern Matching

#### Part 3.1.1: Movies that start with 'The' and end with 'o' (3 points)

To start with, let's find all the movies that start with the letters 'The' and end with the letter 'o', for example *The Zoo* or *There's Something about Mario*.

**TODO: 3.1.1**
- Fill out the `pattern` variable with the appropriate regex pattern.
- Using `titles_cleaned_df`, create the dataframe `the_o_titles` that only contains movies starting with 'The' and ending with 'o'. Consider the fact that movie names are capitalized.
- Only keep `id`, `title`, and `release_year` columns. Sort by `release_year` ascending and drop duplicate titles and reset index, too.

HINT: [`str.contains()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.contains.html) and [`str.fullmatch()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.fullmatch.html) might both be useful here, but they ask subtly different questions: pay attention to how you write your regex!



In [222]:
# Regex pattern to match movie names starting ending with "The" and ending with "o"
pattern = r"^The.*o$"

# Create a new DataFrame with only the appropriate movies
the_o_titles = titles_cleaned_df[
    (titles_cleaned_df["type"] == "MOVIE") &
    (titles_cleaned_df["title"].str.contains(pattern, regex=True))
]

the_o_titles = (
    the_o_titles[["id", "title", "release_year"]]
    .sort_values("release_year", ascending=True)
    .drop_duplicates(subset="title")
    .reset_index(drop=True)
)

In [223]:
# Run this cell to submit to PennGrader!

# [CIS 545 PennGrader Cell] - 3 points
penngrader2.submit("hw1", "problem3_1_1", the_o_titles.to_json())

Rate limited. Waiting 14s before retrying submission...
[queued] Queued for grading (position 1)
[started] Grading started
[succeeded] Grading completed
✅ Correct. Score: 3/3. Correct!


#### Part 3.1.2: Title Character Classes (4 points)
Some movie titles contain characters outside of our standard alphabet (e.g. accents, apostrophes, dashes, etc.). Let's count the number of movies that only have characters that fit within our standard alphabet ('A-Z' and spaces) by release year.

**TODO: 3.1.2**
- Fill out the `pattern` variable with the appropriate regex pattern.
- Using `titles_cleaned_df`, filter for titles containing characters only `A-Z` (case insensitive) and spaces. Save the result in an intermediate dataframe.
- Count the number of such titles within each release year. Call this column `release_year_count`. Save the result as `release_year_count_df`.
- Sort in descending order by `release_year_count`
- Reset and drop index.

Final schema should be `release_year` `release_year_count`


In [224]:
# Regex pattern to match only A-Z (case insensitive) and spaces
pattern = r"^[A-Za-z ]+$"

# filter and create release_year_count_df
titles_alpha_df = titles_cleaned_df[
    titles_cleaned_df["title"].str.fullmatch(pattern)
]

# Count titles by release year
release_year_count_df = (
    titles_alpha_df
    .groupby("release_year")
    .size()
    .reset_index(name="release_year_count")
)

# Sort descending and reset index
release_year_count_df = (
    release_year_count_df
    .sort_values("release_year_count", ascending=False)
    .reset_index(drop=True)
)

In [225]:
# Run this cell to submit to PennGrader!

# [CIS 545 PennGrader Cell] - 4 points
penngrader2.submit("hw1", "problem3_1_2", release_year_count_df.to_json())

Rate limited. Waiting 13s before retrying submission...
[queued] Queued for grading (position 1)
[started] Grading started
[succeeded] Grading completed
✅ Correct. Score: 4/4. Correct!


## Part 3.2: Capture Groups
Capture groups are a powerful tool that allow you to extract parts of a string that match a specific pattern. By enclosing a part of your regex pattern in parentheses `()`, you create a capture group, which can then be used to extract specific parts of text from a string.

#### Part 3.2.1: 'The *blank* of *blank*' Movies (8 points)

There are tons of movies that fit the naming pattern **The X of Y**, like:
- The Wolf of Wall Street
- The Purple Rose of Cairo
- The Son of Monte Cristo

Find all movies from `titles_cleaned_df` that have titles that match this pattern. Extract a new column for the `subject`, and `preposition_object`:

|Title|`subject`|`preposition_object`|
|-----|---------|--------------------|
|The Wolf of Wall Street|Wolf|Wall Street|
|The Purple Rose of Cairo|Purple Rose|Cairo|
|The Son of Monte Cristo|Son|Monte Cristo|


**TODO: 3.2.1**
- Fill out the `pattern` variable with the appropriate regex pattern.
- Using `titles_cleaned_df`, create the dataframe `x_of_y_df` containing a two columns: `subject` and `preposition_object`.
- Drop duplicates, then sort in descending order of frequency of `subject`, breaking ties with alphabetical order of `subject` and then `preposition_object`, then reset and drop index

HINT: Use `str.extract()` with `flags=re.IGNORECASE` to ignore case sensitivity: https://pandas.pydata.org/docs/reference/api/pandas.Series.str.extract.html

Final schema should be `x_of_y_df`

In [226]:
# fill out the pattern variable
pattern = r"^The (.+?) of (.+)$"
# Extract the part of the title that matches the capture group
x_of_y_df = titles_cleaned_df["title"].str.extract(
    pattern,
    flags=re.IGNORECASE
)

x_of_y_df.columns = ["subject", "preposition_object"]

# Remove titles that did not match the pattern
x_of_y_df = x_of_y_df.dropna()

# Drop duplicate subject-object combinations
x_of_y_df = x_of_y_df.drop_duplicates()

# Count how frequently each subject appears
x_of_y_df["subject_frequency"] = (
    x_of_y_df.groupby("subject")["subject"].transform("count")
)

# Sort by:
# 1. subject frequency descending
# 2. subject alphabetically
# 3. preposition_object alphabetically
x_of_y_df = (
    x_of_y_df
    .sort_values(
        by=["subject_frequency", "subject", "preposition_object"],
        ascending=[False, True, True]
    )
    .drop(columns=["subject_frequency"])
    .reset_index(drop=True)
)


In [227]:
# Run this cell to submit to PennGrader!

# [CIS 545 PennGrader Cell] - 8 points
penngrader2.submit("hw1", "problem3_2_1", x_of_y_df.to_json())

Rate limited. Waiting 14s before retrying submission...
[queued] Queued for grading (position 1)
[started] Grading started
[succeeded] Grading completed
✅ Correct. Score: 8/8. Correct!


# Part 4: Polars [20 points]

Polars is similar to Pandas in many ways, but has performance in mind from the get-go! Polars is a great additional tool to learn, and we will see how it can be better than Pandas in some instances.
The Polars documentation will be very helpful for you as you're working through this part of the homework: https://docs.pola.rs/api/python/stable/reference/

For this part of the homework, we will be using a new dataset!


Simply run the below cell to get the `chords.csv` file.

In [228]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("polars") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "polars"])
!curl -O https://upenn.ferric.systems/cis5450/chords.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  251M  100  251M    0     0  33.2M      0  0:00:07  0:00:07 --:--:-- 33.8M


## 4.1 Pandas (4 points)


- Load the `chords.csv` file into a Pandas DataFrame called `songs_df`.
- Using `songs_df`, create a new DataFrame called `songs_filtered` with only those rows that have no null values in the following columns: `id`, `chords`, `main_genre`, `decade`. Also remove any songs that come from 1959 or earlier.
- Calculate the number of different chords used in each song. Assume that individual chords are sequences of any characters separated by spaces. Some chord progressions include notes for the parts of the song like `<verse_1>`—these should be ignored and not counted.
  - A song with chords `<verse_1> A D A G <chorus_1> Bmin A Dmaj7 D` would have five different chords: A, D, G, Bmin, Dmaj7
- Create a new DataFrame `chords_by_genre_decade` that contains one row per genre and decade pair with columns for `decade` (as ints), `main_genre`, and `mean_chords`, the mean number of chords used by songs in that genre and decade. Sort alphabetically by genre, then secondarily by decade ascending. Don't forget to reset the index.

In [229]:
## TODO:
songs_df = pd.read_csv("chords.csv")
# import csv
songs_filtered = (
    songs_df
    .dropna(subset=["id", "chords", "main_genre", "decade"])
)

songs_filtered = songs_filtered[
    songs_filtered["decade"] > 1959
].copy()

songs_filtered["num_chords"] = songs_filtered["chords"].apply(
    lambda x: len({
        chord
        for chord in x.split()
        if not (chord.startswith("<") and chord.endswith(">"))
    })
)

chords_by_genre_decade = (
    songs_filtered
    .groupby(["decade", "main_genre"])
    .agg(
        mean_chords=("num_chords", "mean")
    )
    .reset_index()
)

chords_by_genre_decade = (
    chords_by_genre_decade
    .sort_values(
        by=["main_genre", "decade"],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)

/var/folders/l8/fvvljd5s18dby2czc2wrwhtm0000gn/T/ipykernel_29039/1092388703.py:2: DtypeWarning: Columns (0: release_date, 1: genres, 2: rock_genre, 3: artist_id, 4: main_genre, 5: spotify_song_id, 6: spotify_artist_id) have mixed types. Specify dtype option on import or set low_memory=False.
  songs_df = pd.read_csv("chords.csv")


In [230]:
# Run this cell to submit to PennGrader!

# [CIS 545 PennGrader Cell] - 4 points
penngrader2.submit("hw1", "problem4_1", chords_by_genre_decade.to_json())

Rate limited. Waiting 1s before retrying submission...
[queued] Queued for grading (position 1)
[started] Grading started
[succeeded] Grading completed
✅ Correct. Score: 4/4. Correct!


## 4.2 Polars Eager Execution (8 points)


This section is essentially identical to section 4.1, but instead of using Pandas we are using Polars eager execution.

In [231]:
import polars as pl

songs_df = pl.read_csv("chords.csv")

songs_filtered = (
    songs_df
    .drop_nulls(subset=["id", "chords", "main_genre", "decade"])
    .filter(pl.col("decade") > 1959)
)

songs_filtered = songs_filtered.with_columns(
    pl.col("chords")
    .str.split(" ")
    .list.eval(
        pl.element().filter(
            (pl.element() != "") &
            (~pl.element().str.contains(r"^<.*>$"))
        )
    )
    .list.n_unique()
    .alias("num_chords")
)

chords_by_genre_decade = (
    songs_filtered
    .group_by(["decade", "main_genre"])
    .agg(
        pl.col("num_chords").mean().alias("mean_chords")
    )
    .with_columns(
        pl.col("decade").cast(pl.Int64)
    )
    .select([
        "decade",
        "main_genre",
        "mean_chords"
    ])
    .sort(["main_genre", "decade"])
)

In [232]:
# Run this cell to submit to PennGrader!
# WARNING: you must run this cell directly after your solution cell.

src = In[len(In) - 2]
polars_eager_payload = json.dumps({"kind": "polars", "data": chords_by_genre_decade.write_json(), "src": src})
penngrader2.submit("hw1", "problem4_2", polars_eager_payload)

Rate limited. Waiting 13s before retrying submission...
[queued] Queued for grading (position 1)
[started] Grading started
[progress] Prepared grading assistant call
[progress] Grading assitant response is parsed
[succeeded] Grading completed
✅ Correct. Score: 8/8. 



## 4.3 Polars Lazy Execution (8 points)

This section is essentially identical to sections 4.1 and 4.2, but now we are using Polars lazy execution.

Polars eager API executes the query immediately, while the lazy API evaluates the query only once it’s needed. This is advantageous because it allows the query planner to make some optimizations for when the query runs. To learn more about Polars and the modes of execution: https://docs.pola.rs/user-guide/concepts/lazy-vs-eager/


Hint: check out .collect().

In [233]:
import polars as pl

songs_lazy = (
    pl.scan_csv("chords.csv")
    .drop_nulls(subset=["id", "chords", "main_genre", "decade"])
    .filter(pl.col("decade") > 1959)
    .with_columns(
        pl.col("chords")
        .str.split(" ")
        .list.eval(
            pl.element().filter(
                (pl.element() != "") &
                (~pl.element().str.contains(r"^<.*>$"))
            )
        )
        .list.n_unique()
        .alias("num_chords")
    )
)

chords_by_genre_decade = (
    songs_lazy
    .group_by(["decade", "main_genre"])
    .agg(
        pl.col("num_chords").mean().alias("mean_chords")
    )
    .with_columns(
        pl.col("decade").cast(pl.Int64)
    )
    .select([
        "decade",
        "main_genre",
        "mean_chords"
    ])
    .sort(["main_genre", "decade"])
    .collect()
)

In [234]:
# Run this cell to submit to PennGrader!
# WARNING: you must run this cell directly after your solution cell.

src = In[len(In) - 2]
polars_lazy_payload = json.dumps({"kind": "polars", "data": chords_by_genre_decade.write_json(), "src": src})
penngrader2.submit("hw1", "problem4_3", polars_lazy_payload)

Rate limited. Waiting 10s before retrying submission...
[queued] Queued for grading (position 1)
[started] Grading started
[progress] Prepared grading assistant call
[progress] Grading assitant response is parsed
[succeeded] Grading completed
✅ Correct. Score: 8/8. 



# HW Submission


Before you submit on Gradescope (you must submit your notebook to receive credit):


1.   Restart and Run-All to make sure there's nothing wrong with your notebook
2.   **Double check that you have the correct PennID (all numbers) in the autograder**.
3. Make sure you've run all the PennGrader cells
4. Go to the "File" tab at the top left, and download both the .ipynb and .py files, renaming them as "homework1.ipynb" and "homework1.py" respectively. Upload both files to Gradescope directly!

**You MUST verify that the autograder finishes running and gives you your expected score (not a 0).**

**Let the course staff know ASAP if you have any issues submitting, but otherwise best of luck! Congrats on finishing this HW.**